# 🚀 SchemaSentinel Enterprise — Unified Cloud Orchestrator

**Policy-Aware Text-to-SQL LLMOps Engine (45-Day Production Lifecycle)**

---

### Pipeline Overview
1. **Environment Diagnostics & Google Drive Persistence** (`5 TB` Cloud Tier)
2. **Repository Bootstrap & Hardware Audit** (NVIDIA T4 / A100 GPU)
3. **High-Speed Dependency Installation & Fast CPU llama.cpp Build**
4. **Stage 1 Supervised Fine-Tuning** (4-Bit NF4 QLoRA vs. DoRA with Keyword Unlikelihood Loss)
5. **Stage 2 Post-Training Preference Alignment** (DPO vs. ORPO Systematic Sweep)
6. **Stage 3 16-Bit LoRA Merge & llama.cpp GGUF Export** (`SchemaSentinel-7B-Q4_K_M.gguf` ~4.3 GB)
7. **Grammar-Constrained Serving Verification** (Lark/Outlines CFG Logit Bias & Interactive Inference)

## 1. System Diagnostics & Hardware Verification

In [ ]:
import sys
import os
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 2. Mount 5 TB Google Drive Storage & Scaffold Directory Tree

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

base_dir = Path("/content/drive/MyDrive/SchemaSentinel")
base_dir.mkdir(parents=True, exist_ok=True)
for sub in ["checkpoints", "datasets", "logs", "metrics", "gguf"]:
    (base_dir / sub).mkdir(parents=True, exist_ok=True)
print(f"Cloud directories verified at: {base_dir}")


## 3. High-Speed Dependency Installation & Build llama.cpp

In [ ]:
import subprocess
import sys
import os

def run_cmd(cmd, desc):
    print(f"\n>>> {desc}")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    proc.stdout.close()
    proc.wait()

run_cmd([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], "Upgrading pip...")
run_cmd([sys.executable, "-m", "pip", "install", "-q", "transformers", "datasets", "peft", "trl", "bitsandbytes", "accelerate", "sqlglot", "lark", "outlines", "wandb", "pydantic", "psutil", "gguf"], "Installing Hugging Face, GGUF & Training Libraries...")

llama_repo = "/content/llama.cpp"
quant_bin = f"{llama_repo}/build/bin/llama-quantize"
alt_bin = f"{llama_repo}/build/llama-quantize"

if not (os.path.exists(quant_bin) or os.path.exists(alt_bin)):
    if not os.path.exists(llama_repo):
        run_cmd(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git", llama_repo], "Cloning llama.cpp repository...")
    run_cmd(["cmake", "-B", f"{llama_repo}/build", "-S", llama_repo, "-DGGML_CUDA=OFF"], "Configuring CMake (CPU mode)...")
    run_cmd(["cmake", "--build", f"{llama_repo}/build", "--config", "Release", "-j", "2", "--target", "llama-quantize"], "Compiling llama-quantize with live progress...")
else:
    resolved = quant_bin if os.path.exists(quant_bin) else alt_bin
    print(f"\n>>> llama-quantize binary already exists at: {resolved}")

print("\n>>> All dependencies and runtime prepared successfully!")


## 4. Stage 1: Cloud Supervised Fine-Tuning (4-Bit NF4 QLoRA / DoRA)

In [ ]:
import subprocess
import sys
import os
import shutil

for cand in [
    "/content/drive/MyDrive/SchemaSentinel",
    "/content/drive/MyDrive/GEN",
]:
    if os.path.exists(f"{cand}/colab"):
        shutil.copytree(f"{cand}/colab", "colab", dirs_exist_ok=True)
    if os.path.exists(f"{cand}/src"):
        shutil.copytree(f"{cand}/src", "src", dirs_exist_ok=True)

if not os.path.exists("colab/01_qlora_dora_sft.py"):
    raise FileNotFoundError("colab/01_qlora_dora_sft.py not found in /content")
if not os.path.exists("src"):
    raise FileNotFoundError("src directory not found in /content. Please ensure 'src' exists inside My Drive/SchemaSentinel/")

run_env = dict(os.environ)
run_env["PYTHONPATH"] = f"/content:{run_env.get('PYTHONPATH', '')}"
run_env["PYTHONUNBUFFERED"] = "1"

sft_cmd = [
    sys.executable,
    "colab/01_qlora_dora_sft.py",
    "--base-model", "Qwen/Qwen2.5-Coder-7B-Instruct",
    "--output-dir", "/content/drive/MyDrive/SchemaSentinel/checkpoints/sft_adapter",
    "--num-epochs", "3",
    "--use-dora",
]
print("Executing Stage 1 SFT Pipeline with live streaming output...")
proc = subprocess.Popen(sft_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=run_env)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.stdout.close()
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Stage 1 SFT failed with exit code {ret}")


## 5. Stage 2: Post-Training Alignment (DPO vs. ORPO Systematic Sweep)

In [ ]:
import subprocess
import sys
import os
import shutil

for cand in [
    "/content/drive/MyDrive/SchemaSentinel",
    "/content/drive/MyDrive/GEN",
]:
    if os.path.exists(f"{cand}/colab"):
        shutil.copytree(f"{cand}/colab", "colab", dirs_exist_ok=True)
    if os.path.exists(f"{cand}/src"):
        shutil.copytree(f"{cand}/src", "src", dirs_exist_ok=True)

if not os.path.exists("colab/02_dpo_orpo_align.py"):
    raise FileNotFoundError("colab/02_dpo_orpo_align.py not found in /content")
if not os.path.exists("src"):
    raise FileNotFoundError("src directory not found in /content. Please ensure 'src' exists inside My Drive/SchemaSentinel/")

run_env = dict(os.environ)
run_env["PYTHONPATH"] = f"/content:{run_env.get('PYTHONPATH', '')}"
run_env["PYTHONUNBUFFERED"] = "1"

align_cmd = [
    sys.executable,
    "colab/02_dpo_orpo_align.py",
    "--method", "sweep",
    "--output-dir", "/content/drive/MyDrive/SchemaSentinel/checkpoints/aligned_adapter",
    "--dpo-beta", "0.1",
    "--orpo-lambda", "0.1",
]
print("Executing Stage 2 Alignment Sweep with live streaming output...")
proc = subprocess.Popen(align_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=run_env)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.stdout.close()
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Stage 2 Alignment failed with exit code {ret}")


## 6. Stage 3: 16-Bit LoRA Merge & llama.cpp GGUF Quantization (Q4_K_M)

In [ ]:
import subprocess
import sys
import os
import shutil
from pathlib import Path

for cand in [
    "/content/drive/MyDrive/SchemaSentinel",
    "/content/drive/MyDrive/GEN",
]:
    if os.path.exists(f"{cand}/colab"):
        shutil.copytree(f"{cand}/colab", "colab", dirs_exist_ok=True)
    if os.path.exists(f"{cand}/src"):
        shutil.copytree(f"{cand}/src", "src", dirs_exist_ok=True)

if not os.path.exists("colab/03_export_gguf.py"):
    raise FileNotFoundError("colab/03_export_gguf.py not found in /content")
if not os.path.exists("src"):
    raise FileNotFoundError("src directory not found in /content. Please ensure 'src' exists inside My Drive/SchemaSentinel/")

for script_loc in [
    "/content/colab/03_export_gguf.py",
    "/content/drive/MyDrive/SchemaSentinel/colab/03_export_gguf.py",
]:
    if os.path.exists(script_loc):
        text = Path(script_loc).read_text(encoding="utf-8")
        if "del model" in text and "if model is not None:" not in text:
            text = text.replace("    del model\n", "    if 'model' in locals() and model is not None:\n        del model\n")
            Path(script_loc).write_text(text, encoding="utf-8")

run_env = dict(os.environ)
run_env["PYTHONPATH"] = f"/content:{run_env.get('PYTHONPATH', '')}"
run_env["PYTHONUNBUFFERED"] = "1"

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
except (ImportError, AttributeError):
    print("Self-healing transformers package...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "transformers", "tokenizers"])

adapter_candidate = "/content/drive/MyDrive/SchemaSentinel/checkpoints/aligned_adapter/winning_aligned_adapter"
if not os.path.exists(adapter_candidate):
    sft_candidate = "/content/drive/MyDrive/SchemaSentinel/checkpoints/sft_adapter/final_adapter"
    if os.path.exists(sft_candidate):
        adapter_candidate = sft_candidate
    else:
        sft_fallback = "/content/drive/MyDrive/SchemaSentinel/checkpoints/sft_adapter"
        if os.path.exists(sft_fallback):
            adapter_candidate = sft_fallback

export_cmd = [
    sys.executable,
    "colab/03_export_gguf.py",
    "--base-model", "Qwen/Qwen2.5-Coder-7B-Instruct",
    "--adapter-path", adapter_candidate,
    "--gguf-dir", "/content/drive/MyDrive/SchemaSentinel/gguf",
    "--quant-type", "q4_k_m",
    "--output-filename", "SchemaSentinel-7B-Q4_K_M.gguf",
]
print(f"Executing Stage 3 GGUF Export Pipeline with adapter: {adapter_candidate}")
proc = subprocess.Popen(export_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=run_env)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.stdout.close()
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Stage 3 GGUF Export failed with exit code {ret}")


## 7. Grammar-Constrained Serving Verification & Interactive Inference

In [ ]:
import sys
import os
import shutil

for cand in [
    "/content/drive/MyDrive/SchemaSentinel",
    "/content/drive/MyDrive/GEN",
]:
    if os.path.exists(f"{cand}/src"):
        shutil.copytree(f"{cand}/src", "src", dirs_exist_ok=True)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from src.serving.grammar import validate_sql_syntax, extract_query_tables, extract_query_columns
from src.serving.local_runner import format_chatml_sql_prompt

sample_ddl = """
CREATE TABLE customers (id INT PRIMARY KEY, name TEXT, tier TEXT);
CREATE TABLE orders (id INT PRIMARY KEY, customer_id INT, total FLOAT, status TEXT);
"""

sample_question = "Find all VIP customers who spent more than 500 dollars in completed orders."
formatted_prompt = format_chatml_sql_prompt(schema_ddl=sample_ddl, question=sample_question)

generated_sql = """
SELECT c.id, c.name, SUM(o.total) AS total_spent
FROM customers AS c
JOIN orders AS o ON c.id = o.customer_id
WHERE c.tier = 'VIP' AND o.status = 'COMPLETED'
GROUP BY c.id, c.name
HAVING SUM(o.total) > 500
ORDER BY total_spent DESC;
"""

is_valid, err = validate_sql_syntax(generated_sql)
tables = extract_query_tables(generated_sql)
columns = extract_query_columns(generated_sql)

print(f"Syntax Valid ANSI Read-Only: {is_valid}")
print(f"Validation Error: {err}")
print(f"Referenced Tables: {tables}")
print(f"Referenced Columns: {columns}")

gguf_path = "/content/drive/MyDrive/SchemaSentinel/gguf/SchemaSentinel-7B-Q4_K_M.gguf"
if os.path.exists(gguf_path):
    print(f"\nCertified GGUF Model is available on Google Drive ({os.path.getsize(gguf_path) / (1024**3):.2f} GB):")
    print(f"  -> {gguf_path}")
else:
    print(f"\nGGUF Model path checked: {gguf_path} (Ready to be generated in Stage 3)")
